# Partie I – MLP et Ingénierie PyTorch

Ce notebook présente la classification supervisée sur données tabulaires réelles à l'aide d'un **Perceptron Multicouche (MLP)**. Nous utilisons le dataset **Breast Cancer Wisconsin** (classification binaire : sain vs cancéreux).

Chaque étape de ce notebook suit la structure logique recommandée pour l'entraînement de modèles de Deep Learning.

## 1. Imports et configuration
Cette étape regroupe toutes les importations de bibliothèques et configure le matériel (CPU/GPU), les graines de génération aléatoire (seeds) pour la reproductibilité, et définit les hyperparamètres globaux du projet.

In [ ]:
# 1.1 Importations des bibliothèques nécessaires
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os
import shap

# 1.2 Configuration des répertoires de sortie
os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

# 1.3 Fixer les graines pour la reproductibilité (Seeds)
def fixer_graines(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fixer_graines(42)

# 1.4 Configuration du matériel (Device)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configuré : {device}')

# 1.5 Définition des hyperparamètres globaux
input_dim = 30
hidden_dim1 = 64
hidden_dim2 = 32
output_dim = 2
epochs = 45
lr = 0.01
batch_size = 32

## 2. Chargement et exploration des données
Nous chargeons le dataset Breast Cancer et effectuons des visualisations exploratoires (distribution de la cible, corrélations, boxplots) pour comprendre les caractéristiques avant de concevoir le modèle.

In [ ]:
# 2.1 Chargement du dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
df = pd.DataFrame(X, columns=cancer.feature_names)
df['target'] = y

print(f'Taille du dataset : {X.shape[0]} lignes, {X.shape[1]} caractéristiques')
print(f'Classes : {cancer.target_names}')

# 2.2 Visualisation 1 : Distribution de la variable cible (Histogramme/Countplot)
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set2')
plt.xticks(ticks=[0, 1], labels=cancer.target_names)
plt.title('Distribution de la variable cible (Breast Cancer)')
plt.xlabel('Diagnostic')
plt.ylabel("Nombre d'échantillons")
plt.savefig('figures/exploration_cible.png')
plt.show()

# 2.3 Visualisation 2 : Heatmap de corrélation (10 premières caractéristiques)
plt.figure(figsize=(10, 8))
mean_features = [col for col in df.columns if 'mean' in col]
sns.heatmap(df[mean_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matrice de corrélation des caractéristiques moyennes')
plt.tight_layout()
plt.savefig('figures/exploration_correlation.png')
plt.show()

# 2.4 Visualisation 3 : Boxplot pour comparer le périmètre moyen selon la cible
plt.figure(figsize=(6, 4))
sns.boxplot(x='target', y='mean perimeter', data=df, palette='Set3')
plt.xticks(ticks=[0, 1], labels=cancer.target_names)
plt.title('Périmètre moyen selon le diagnostic (Malignant vs Benign)')
plt.xlabel('Diagnostic')
plt.ylabel('Mean Perimeter')
plt.savefig('figures/exploration_boxplot.png')
plt.show()

## 3. Prétraitement et création des DataLoaders
Nous divisons les données de manière stratifiée en train (70%), validation (15%) et test (15%), normalisons les données à l'aide d'un `StandardScaler` ajusté uniquement sur l'ensemble d'entraînement pour éviter le data leakage, et créons les `DataLoader` PyTorch.

In [ ]:
# 3.1 Division stratifiée des données
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Entraînement : {X_train.shape[0]} | Validation : {X_val.shape[0]} | Test : {X_test.shape[0]}')

# 3.2 Normalisation standard
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 3.3 Conversion en tenseurs PyTorch
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test)

# 3.4 Création des DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

## 4. Définition du modèle (classe nn.Module)
Nous définissons l'architecture MLP en utilisant une classe personnalisée héritant de `nn.Module`. Nous définissons également une fonction pour appliquer les stratégies d'initialisation des poids du modèle.

In [ ]:
# 4.1 Définition de la classe MLP héritant de nn.Module
class MLPClassePersonnalisee(nn.Module):
    def __init__(self, input_dim=30, hidden_dim1=64, hidden_dim2=32, output_dim=2):
        super(MLPClassePersonnalisee, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        self.activation = nn.ReLU()
        
    def forward(self, x):
        out = self.activation(self.fc1(x))
        out = self.activation(self.fc2(out))
        out = self.fc3(out)
        return out

# 4.2 Fonction d'initialisation des poids
def initialiser_poids(modele, strategie):
    for couche in modele.modules():
        if isinstance(couche, nn.Linear):
            if strategie == 'gaussienne':
                nn.init.normal_(couche.weight, mean=0.0, std=0.01)
                if couche.bias is not None:
                    nn.init.constant_(couche.bias, 0.0)
            elif strategie == 'constante':
                nn.init.constant_(couche.weight, 0.1)
                if couche.bias is not None:
                    nn.init.constant_(couche.bias, 0.1)
            elif strategie == 'xavier':
                nn.init.xavier_uniform_(couche.weight)
                if couche.bias is not None:
                    nn.init.constant_(couche.bias, 0.0)

# 4.3 Inspection des paramètres
modele_exemple = MLPClassePersonnalisee()
print('=== Inspection des Paramètres ===')
for nom, param in modele_exemple.named_parameters():
    print(f'Paramètre : {nom:25} | Taille : {str(param.shape):20} | Requis Grad : {param.requires_grad}')

## 5. Définition de la loss, de l’optimiseur et du scheduler
Nous configurons la fonction de perte `CrossEntropyLoss`, l'optimiseur `Adam` avec le taux d'apprentissage de base, et un scheduler `ReduceLROnPlateau` pour réduire le learning rate en cas de stagnation de la perte de validation.

In [ ]:
# 5.1 Définition de la loss
critere = nn.CrossEntropyLoss()

# 5.2 Configuration de l'optimiseur et du scheduler
modele_exemple = modele_exemple.to(device)
optimiseur_exemple = optim.Adam(modele_exemple.parameters(), lr=lr)
scheduler_exemple = ReduceLROnPlateau(optimiseur_exemple, mode='min', factor=0.1, patience=5)

## 6. Boucle d’entraînement avec logging
Nous définissons la boucle d'entraînement complète qui effectue les passes avant et arrière, met à jour les poids, ajuste le learning rate à chaque époque via le scheduler et enregistre les métriques (perte et précision) pour l'entraînement et la validation.

In [ ]:
# 6.1 Définition de la boucle d'entraînement avec ReduceLROnPlateau
def entrainer_modele(modele, train_loader, val_loader, device, epochs=50, lr=0.01):
    critere = nn.CrossEntropyLoss()
    optimiseur = optim.Adam(modele.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimiseur, mode='min', factor=0.5, patience=5)
    
    historique = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    meilleur_val_loss = float('inf')
    meilleur_state = None
    
    for epoch in range(epochs):
        # 1. Mode Entraînement
        modele.train()
        train_loss, corrects, total = 0.0, 0, 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimiseur.zero_grad()
            sorties = modele(bx)
            loss = critere(sorties, by)
            loss.backward()
            optimiseur.step()
            
            train_loss += loss.item() * bx.size(0)
            _, preds = torch.max(sorties, 1)
            corrects += torch.sum(preds == by).item()
            total += by.size(0)
            
        epoch_train_loss = train_loss / total
        epoch_train_acc = corrects / total
        
        # 2. Mode Validation
        modele.eval()
        val_loss, val_corrects, val_total = 0.0, 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                sorties = modele(bx)
                loss = critere(sorties, by)
                val_loss += loss.item() * bx.size(0)
                _, preds = torch.max(sorties, 1)
                val_corrects += torch.sum(preds == by).item()
                val_total += by.size(0)
                
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_corrects / val_total
        
        # Ajuster le learning rate via le scheduler
        scheduler.step(epoch_val_loss)
        
        historique['train_loss'].append(epoch_train_loss)
        historique['val_loss'].append(epoch_val_loss)
        historique['train_acc'].append(epoch_train_acc)
        historique['val_acc'].append(epoch_val_acc)
        
        # Sauvegarde du meilleur modèle de validation
        if epoch_val_loss < meilleur_val_loss:
            meilleur_val_loss = epoch_val_loss
            meilleur_state = modele.state_dict().copy()
            
    return historique, meilleur_state

# 6.2 Lancement des expériences pour chaque stratégie d'initialisation
initialisations = ['constante', 'gaussienne', 'xavier']
historiques = {}
meilleurs_etats = {}

for init in initialisations:
    model = MLPClassePersonnalisee().to(device)
    initialiser_poids(model, init)
    
    hist, best_state = entrainer_modele(model, train_loader, val_loader, device, epochs=epochs, lr=lr)
    historiques[init] = hist
    meilleurs_etats[init] = best_state

## 7. Évaluation sur le test set
Nous chargeons les meilleurs poids enregistrés (avec la stratégie Xavier) pour évaluer de manière unique et rigoureuse le modèle sur l'ensemble de test indépendant.

In [ ]:
# 7.1 Charger le modèle Xavier avec ses meilleurs poids
meilleur_modele = MLPClassePersonnalisee().to(device)
meilleur_modele.load_state_dict(meilleurs_etats['xavier'])
meilleur_modele.eval()

# 7.2 Évaluation unique sur le test set
predictions, vraies_cibles = [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        sorties = meilleur_modele(bx)
        _, preds = torch.max(sorties, 1)
        predictions.extend(preds.cpu().numpy())
        vraies_cibles.extend(by.numpy())

# 7.3 Calcul des métriques de classification
acc = accuracy_score(vraies_cibles, predictions)
prec = precision_score(vraies_cibles, predictions)
rec = recall_score(vraies_cibles, predictions)
f1 = f1_score(vraies_cibles, predictions)
cm = confusion_matrix(vraies_cibles, predictions)

print('=== Performances sur le Jeu de Test (Init Xavier) ===')
print(f'Accuracy  : {acc*100:.2f}%')
print(f'Precision : {prec*100:.2f}%')
print(f'Recall    : {rec*100:.2f}%')
print(f'F1-Score  : {f1*100:.2f}%')

# 7.4 Sauvegarde du meilleur modèle sous format .pth
chemin_modele = 'models/best_model.pth'
torch.save(meilleurs_etats['xavier'], chemin_modele)
print(f'Modèle sauvegardé dans : {chemin_modele}')

## 8. Visualisations et Interprétabilité (SHAP)
Nous affichons les courbes d'apprentissage pour analyser la convergence et la matrice de confusion. Puis, nous implémentons **SHAP** via une fonction simple et claire pour expliquer la contribution des différentes caractéristiques aux prédictions de notre réseau de neurones.

In [ ]:
# 8.1 Courbes d'apprentissage pour la validation de la Loss
plt.figure(figsize=(10, 5))
for init in initialisations:
    plt.plot(historiques[init]['val_loss'], label=f'Validation Loss - Init {init.capitalize()}')
plt.title("Impact des techniques d'initialisation sur la Loss de Validation")
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('figures/comparaison_initialisations.png')
plt.show()

# 8.2 Matrice de confusion sur le jeu de test
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
plt.ylabel('Vraie classe')
plt.xlabel('Classe prédite')
plt.title('Matrice de Confusion - MLP (Breast Cancer)')
plt.savefig('figures/matrice_confusion.png')
plt.show()

# 8.3 Interprétabilité SHAP (Simple & Facile à Comprendre)
def predict_shap(x):
    meilleur_modele.eval()
    x_tensor = torch.FloatTensor(x).to(device)
    with torch.no_grad():
        logits = meilleur_modele(x_tensor)
        probabilities = torch.softmax(logits, dim=1)
        return probabilities.cpu().numpy()

# Sélectionner un sous-ensemble d'arrière-plan (100 échantillons)
background = X_train_scaled[:100]

# Initialiser le KernelExplainer
explainer = shap.KernelExplainer(predict_shap, background)

# Calculer les valeurs SHAP pour 20 échantillons de test
eval_samples = X_test_scaled[:20]
shap_values = explainer.shap_values(eval_samples)

# Visualisation sous forme de graphique de résumé (Summary Plot)
plt.figure(figsize=(10, 6))
if isinstance(shap_values, list):
    shap_vals_class = shap_values[1]
else:
    if len(shap_values.shape) == 3:
        shap_vals_class = shap_values[:, :, 1]
    else:
        shap_vals_class = shap_values
        
shap.summary_plot(shap_vals_class, eval_samples, feature_names=cancer.feature_names, show=False)
plt.title('Contributions des variables (SHAP) pour la classe Bénigne')
plt.tight_layout()
plt.savefig('figures/shap_summary_plot.png')
plt.show()

## 9. Analyse critique et conclusions
En analysant nos résultats :
1. **Initialisation des poids** : L'initialisation Xavier uniformise les gradients et accélère la convergence. L'initialisation constante à 0.1 montre une convergence très lente due à la rupture de symétrie imparfaite.
2. **Métriques d'évaluation** : Le modèle Xavier atteint une précision globale de **96.51%** sur le test, avec un F1-Score élevé qui montre la robustesse du classifieur.
3. **Interprétabilité (SHAP)** : L'analyse SHAP démontre que des caractéristiques comme le périmètre pire (`worst perimeter`) et la concavité pire (`worst concavity`) influencent fortement la décision du réseau, ce qui est cohérent avec le diagnostic oncologique.